# Yelp Aspect Agents: the whole pipeline in one notebook

A team of LLM agents reads Yelp restaurant reviews and judges three aspects: **food**, **service** and
**ambience**. The agents only quote the review and label each quote; **every number is computed in Python**.
This notebook then does two things:

1. **Accuracy check**: 100 reviews (20 per star rating), compared with the reviewers' stars.
2. **Restaurant comparison**: 50 random restaurants × 10 reviews each, to see where reviewers of the *same*
   restaurant agree or disagree.

It uses the same model as the project, **`gemma4-31b-it` on ASU's Voyager LLM API**
(`https://openai.rc.asu.edu/v1`), and the project's prompts, quote checking, formulas, samplers and statistics.
With seed 42 it draws the same reviews as `yelp-agents sample` and `yelp-agents sample-restaurants`.

**Before you run it**

1. Create a Voyager API key. Connect to the ASU VPN (`sslvpn.asu.edu/2fa`), sign in at
   [voyager.rc.asu.edu](https://voyager.rc.asu.edu), open **LLM Access** and click **Create Key**.
2. In Colab, click the **key icon** in the left sidebar, add a secret named `VOYAGER_API_KEY` with your key,
   and turn on **Notebook access**. Never paste the key into a code cell.
3. Upload `yelp_dataset.tar` (the **Download JSON** file from the
   [Yelp Open Dataset](https://business.yelp.com/data/resources/open-dataset/), 4.35 GB) to the top of your Google
   Drive, or use one of the other data options in section 2.

Then use **Runtime → Run all**. The full run analyses 600 reviews: about 15 minutes with 4 workers, plus a few
minutes to read the tar from Drive. For a quick first pass, set `MAX_REVIEWS_TO_ANALYZE` to 20 in section 2.

| Section | What happens |
| --- | --- |
| 1. Setup | Install, read your key, check the connection |
| 2. Reviews | Sample from the Yelp tar, or load a sample |
| 3. Prompts | Exactly what each agent is told |
| 4. Quote checking and scoring | Pure Python, with quick self-checks |
| 5. The agents | Aspect agents, arbiter, lead agent |
| 6. One review, step by step | Every quote and the arithmetic behind each score |
| 7. Accuracy check | Run 100 reviews and compare with the stars |
| 8. Restaurant comparison | Run 50 × 10 reviews and compare reviewers of each restaurant |
| 9. Exercises | Things to try next |

## 1. Setup

Colab already has pandas, NumPy, SciPy, scikit-learn, Matplotlib and Plotly; only the OpenAI client library is
installed here. Voyager speaks the same API as OpenAI, so the official client works with a different base URL.

In [ ]:
%pip install -q openai

In [ ]:
import concurrent.futures as cf
import getpass
import json
import math
import os
import random
import re
import tarfile
import time
import zlib
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from openai import OpenAI
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             mean_absolute_error, mean_squared_error)
from sklearn.model_selection import KFold, cross_val_predict
from tqdm.auto import tqdm

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_URL = "https://openai.rc.asu.edu/v1"   # ASU Research Computing's OpenAI-compatible LLM API
MODEL = "gemma4-31b-it"  #@param ["gemma4-31b-it", "qwen3-235b-a22b-instruct-2507", "llama4-maverick-17b"] {allow-input: true}
TEMPERATURE = 0.1        # low, so a re-run gives nearly the same answers
WORKERS = 4  #@param {type:"integer"}
USE_FEW_SHOT = True  #@param {type:"boolean"}

ASPECTS = ("food", "service", "ambience")
POLARITY_SIGN = {"positive": 1, "negative": -1, "neutral": 0}
CLASSES = ["negative", "neutral", "positive"]

In [ ]:
def get_api_key() -> str:
    """The Colab secret VOYAGER_API_KEY, else an environment variable, else a hidden prompt."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("VOYAGER_API_KEY")
            if key:
                return key.strip()
        except Exception as exc:  # secret missing, or notebook access not switched on
            print(f"No usable Colab secret ({type(exc).__name__}); asking for the key instead.")
    key = os.environ.get("VOYAGER_API_KEY") or os.environ.get("API_KEY")
    return (key or getpass.getpass("Voyager API key: ")).strip()


# The client retries rate limits (HTTP 429), server errors and timeouts by itself.
client = OpenAI(base_url=BASE_URL, api_key=get_api_key(), max_retries=4, timeout=120)

try:
    available = sorted(m.id for m in client.models.list())
except Exception as exc:
    print("Could not reach the Voyager API:", type(exc).__name__, str(exc)[:200])
    print("- 401 Unauthorized: the key is wrong or revoked. Create a new one in Voyager's LLM Access tab.")
    print("- Connection error or timeout: run the notebook on your own machine with the ASU VPN connected,")
    print("  or connect Colab to a local runtime (Connect > Connect to a local runtime).")
    raise
print(f"Connected: {len(available)} models available.")
print(f"{MODEL}: {'available' if MODEL in available else 'NOT available, pick another model'}")

## 2. Get the reviews

Pick one data source:

| `DATA_SOURCE` | Use it when |
| --- | --- |
| `tar` | `yelp_dataset.tar` is in your Google Drive (or anywhere this notebook can read). One pass over all 6,990,280 reviews draws both samples; from Drive this takes a few minutes. |
| `jsonl` | You already have `sample_reviews.jsonl` and `restaurant_reviews.jsonl` from `yelp-agents sample` / `sample-restaurants`. In Colab you are asked to upload them if they're missing. |
| `demo` | You just want to see the pipeline run. Eight short **made-up** reviews of two made-up restaurants, no dataset needed. |

The samples are saved next to the results, so a later session loads them instead of scanning again. With
`SAVE_TO_DRIVE` on, results go to `My Drive/yelp_aspect_agents/` and survive a Colab disconnect.

In [ ]:
DATA_SOURCE = "tar"  #@param ["tar", "jsonl", "demo"]
TAR_PATH = "/content/drive/MyDrive/yelp_dataset.tar"  #@param {type:"string"}
SAMPLE_JSONL = "sample_reviews.jsonl"  #@param {type:"string"}
RESTAURANT_JSONL = "restaurant_reviews.jsonl"  #@param {type:"string"}
N_REVIEWS = 100  #@param {type:"integer"}
N_RESTAURANTS = 50  #@param {type:"integer"}
REVIEWS_PER_RESTAURANT = 10  #@param {type:"integer"}
MAX_REVIEWS_TO_ANALYZE = 0  #@param {type:"integer"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
SEED = 42  # the project's seed: the same seed draws the same reviews

if IN_COLAB and (SAVE_TO_DRIVE or TAR_PATH.startswith("/content/drive")):
    from google.colab import drive
    drive.mount("/content/drive")

OUT_DIR = Path("/content/drive/MyDrive/yelp_aspect_agents") if IN_COLAB and SAVE_TO_DRIVE else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results folder:", OUT_DIR.resolve())

**How the samples are drawn.** Both samplers use *reservoir sampling* (Algorithm R): to keep k items from a
stream of unknown length, keep the first k, then replace slot j with the n-th item whenever a random j in
0…n−1 is below k. Every item ends up with the same chance of being kept, in a single pass and without
extracting the archive. Only reviews of businesses in the Restaurants category, 40–2,000 characters long, count.

- **Accuracy sample**: one reservoir per star rating, 20 reviews each, then shuffled.
- **Restaurant sample**: a reservoir of 150 candidate restaurants (each listing at least 10 reviews), one
  reservoir of reviews per candidate, then the first 50 candidates, in random order, that have 10 usable reviews.

In [ ]:
BUSINESS_MEMBER = "yelp_academic_dataset_business.json"
REVIEW_MEMBER = "yelp_academic_dataset_review.json"
_BID_KEY = b'"business_id":"'


class Reservoir:
    """A uniform random sample of k items from a stream of unknown length (Algorithm R)."""

    def __init__(self, k, rng):
        self.k, self.rng, self.items, self.seen = k, rng, [], 0

    def offer(self, item):
        self.seen += 1
        if len(self.items) < self.k:
            self.items.append(item)
        else:
            j = self.rng.randrange(self.seen)
            if j < self.k:
                self.items[j] = item


def _lines(tar, info):
    for raw in tar.extractfile(info):
        if raw.strip():
            yield raw


def _is_restaurant(b):
    return "restaurants" in [c.strip().lower() for c in (b.get("categories") or "").split(",")]


def _business_id(raw):
    """Read business_id straight from the raw line, so most reviews are never parsed as JSON."""
    i = raw.find(_BID_KEY)
    if i < 0:
        return None
    return raw[i + len(_BID_KEY):raw.find(b'"', i + len(_BID_KEY))].decode()


def _usable(r):
    return 1 <= int(round(float(r["stars"]))) <= 5 and 40 <= len((r.get("text") or "").strip()) <= 2000


def _review(r, b):
    return {"review_id": r["review_id"], "business_id": r["business_id"], "stars": float(round(float(r["stars"]))),
            "text": r["text"].strip(), "business_name": b.get("name", ""), "city": b.get("city", ""),
            "categories": b.get("categories", ""), "date": r.get("date", ""),
            "business_stars": b.get("stars"), "business_review_count": b.get("review_count")}


def sample_from_tar(tar_path, n_reviews, n_restaurants, per_restaurant, seed=42, oversample=3):
    """Both samples in one streaming pass. Each sampler has its own random generator, so the result
    matches `yelp-agents sample` and `yelp-agents sample-restaurants` run separately with the same seed."""
    rng_a, rng_b = random.Random(seed), random.Random(seed)
    by_star = {s: Reservoir(max(1, n_reviews // 5), rng_a) for s in range(1, 6)}
    candidates = Reservoir(n_restaurants * oversample, rng_b)
    restaurants, per_candidate = {}, {}
    with tarfile.open(tar_path, mode="r|*") as tar:  # stream mode: read front to back, no seeking
        for info in tar:
            name = Path(info.name).name
            if name == BUSINESS_MEMBER:  # comes before the reviews inside the archive
                for raw in _lines(tar, info):
                    b = json.loads(raw)
                    if _is_restaurant(b):
                        restaurants[b["business_id"]] = b
                        if (b.get("review_count") or 0) >= per_restaurant:
                            candidates.offer(b)
                per_candidate = {b["business_id"]: Reservoir(per_restaurant, rng_b) for b in candidates.items}
                print(f"{len(restaurants):,} restaurants; {candidates.seen:,} list at least {per_restaurant} reviews")
            elif name == REVIEW_MEMBER:
                for raw in tqdm(_lines(tar, info), desc="scanning reviews", unit=" reviews", mininterval=2):
                    bid = _business_id(raw)
                    if bid not in restaurants:
                        continue
                    r = json.loads(raw)
                    if not _usable(r):
                        continue
                    by_star[int(round(float(r["stars"])))].offer(r)
                    if bid in per_candidate:
                        per_candidate[bid].offer(r)
                break
    accuracy = [_review(r, restaurants[r["business_id"]]) for s in range(1, 6) for r in by_star[s].items]
    rng_a.shuffle(accuracy)
    order = list(candidates.items)
    rng_b.shuffle(order)
    chosen = [b for b in order if per_candidate[b["business_id"]].seen >= per_restaurant][:n_restaurants]
    grouped = [_review(r, b) for b in chosen  # grouped by restaurant, oldest review first
               for r in sorted(per_candidate[b["business_id"]].items, key=lambda r: r.get("date", ""))]
    return accuracy, grouped


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

In [ ]:
# Made-up reviews, only for DATA_SOURCE = "demo". They are not from Yelp.
DEMO_REVIEWS = [
    ("Demo Diner", 5, "The pancakes were fluffy and the coffee kept coming. Our server remembered our order from last week. Bright, cheerful room."),
    ("Demo Diner", 4, "Great omelette and crispy hash browns. We waited twenty minutes on a Sunday but the staff were apologetic."),
    ("Demo Diner", 2, "The eggs were cold and the toast was burnt. Nobody came back to check on us for half an hour."),
    ("Demo Diner", 4, "Solid breakfast, generous portions. It gets very loud when it's busy, so not a place to talk."),
    ("Demo Bistro", 1, "Rude host, and we were seated next to the kitchen door. The soup was lukewarm and oversalted."),
    ("Demo Bistro", 3, "The steak was cooked well but the sides were bland. Service was fine. Dim, romantic lighting."),
    ("Demo Bistro", 2, "Overpriced pasta that tasted like it came from a jar. Our waiter was friendly though."),
    ("Demo Bistro", 5, "Lovely candlelit room and a wine list to match. The duck was perfect and our waitress was attentive."),
]


def demo_reviews():
    out = []
    for i, (name, stars, text) in enumerate(DEMO_REVIEWS):
        out.append({"review_id": f"demo-{i}", "business_id": name.lower().replace(" ", "-"), "stars": float(stars),
                    "text": text, "business_name": name, "city": "Nowhere", "categories": "Restaurants",
                    "date": f"2024-01-{i + 1:02d}", "business_stars": None, "business_review_count": None})
    return out


acc_path = OUT_DIR / f"sample_{N_REVIEWS}_seed{SEED}.jsonl"
rest_path = OUT_DIR / f"restaurants_{N_RESTAURANTS}x{REVIEWS_PER_RESTAURANT}_seed{SEED}.jsonl"

if DATA_SOURCE == "demo":
    accuracy_sample = restaurant_sample = demo_reviews()
elif DATA_SOURCE == "jsonl":
    if IN_COLAB and not (Path(SAMPLE_JSONL).exists() and Path(RESTAURANT_JSONL).exists()):
        from google.colab import files
        print("Upload", SAMPLE_JSONL, "and", RESTAURANT_JSONL)
        files.upload()
    accuracy_sample, restaurant_sample = read_jsonl(SAMPLE_JSONL), read_jsonl(RESTAURANT_JSONL)
elif acc_path.exists() and rest_path.exists():
    accuracy_sample, restaurant_sample = read_jsonl(acc_path), read_jsonl(rest_path)
    print("Loaded the samples saved by an earlier session.")
else:
    t0 = time.time()
    accuracy_sample, restaurant_sample = sample_from_tar(TAR_PATH, N_REVIEWS, N_RESTAURANTS, REVIEWS_PER_RESTAURANT, SEED)
    write_jsonl(accuracy_sample, acc_path)
    write_jsonl(restaurant_sample, rest_path)
    print(f"Sampled in {time.time() - t0:.0f} s.")

print(f"Accuracy sample: {len(accuracy_sample)} reviews, stars {dict(sorted(Counter(int(r['stars']) for r in accuracy_sample).items()))}")
print(f"Restaurant sample: {len(restaurant_sample)} reviews of {len({r['business_id'] for r in restaurant_sample})} restaurants")
pd.DataFrame(accuracy_sample)[["business_name", "city", "stars", "text"]].head()

## 3. The prompts

These are the project's prompts, copied unchanged from `packages/yelp-agents/src/yelp_agents/prompts.py`.
Read them: they are the whole specification the agents work from.

- Each **aspect agent** gets its aspect's definition (with what does *not* count), rules for what to quote,
  the intensity rubric, and 2–3 hand-written examples. The examples are deliberately not from Yelp, so no test
  review leaks into the prompt.
- The **arbiter** decides which aspect a passage belongs to when two aspect agents quote the same words.
- The **lead agent** reads the full review plus the quotes the aspect agents kept, and gives the overall verdict.

In [ ]:
ASPECT_DEFINITIONS = {
    "food": "the food and drink itself: taste, freshness, temperature, portion size, menu variety, presentation. "
            "NOT how fast it arrived (service) and NOT its price.",
    "service": "the staff and the service process: friendliness, attentiveness, speed / wait times, order accuracy, "
               "how problems were handled. NOT the room or the food itself.",
    "ambience": "the physical setting: atmosphere, decor, noise level, music, cleanliness of the premises, seating, "
                "comfort, vibe. NOT staff behaviour, NOT waiting times, NOT the food.",
}

INTENSITY_RUBRIC = """intensity:
  1 = mild or hedged ("decent", "a bit slow", "okay but nothing special")
  2 = clear ("good", "friendly", "slow", "cold fries")
  3 = strong or emphatic ("amazing", "best I've had", "disgusting", "never coming back because of the staff")"""

ASPECT_SYSTEM = """You are the {aspect_upper} specialist in a team of agents analysing Yelp reviews.
You ONLY look at {aspect}: {definition}
Other agents cover the other aspects; ignore them. Price and location are not {aspect}.

List every statement in which the reviewer JUDGES the {aspect} of this business.
Include only evaluative statements. Skip plain descriptions of what was ordered, background
story, and wishes for other customers. Read polarity from the reviewer's point of view,
including sarcasm and counterfactuals ("I'd have been better off ordering X" is negative about
what they got; "I hope others have a better experience" is negative).
For each statement give:
  quote: copied EXACTLY from the review (verbatim, at least two words, one sentence or clause)
  polarity: "positive" | "negative" | "neutral"
{rubric}

If the review says nothing about {aspect}, return an empty list.
Return a single JSON object and nothing else:
{{"mentions": [{{"quote": "...", "polarity": "...", "intensity": 1}}]}}"""

FEW_SHOT = {
    "food": [
        ("The tacos were bland and the tortillas stale, but our server was a sweetheart. The salsa was incredible though!",
         '{"mentions": [{"quote": "The tacos were bland and the tortillas stale", "polarity": "negative", "intensity": 2}, '
         '{"quote": "The salsa was incredible though!", "polarity": "positive", "intensity": 3}]}'),
        ("Great patio and friendly staff. Can't wait to come back!", '{"mentions": []}'),
    ],
    "service": [
        ("Food was incredible. We waited 45 minutes for a table and nobody apologized.",
         '{"mentions": [{"quote": "We waited 45 minutes for a table and nobody apologized.", "polarity": "negative", "intensity": 3}]}'),
        ("Cozy little spot, the pho is the best in town.", '{"mentions": []}'),
    ],
    "ambience": [
        ("Super loud inside and the tables were sticky, though the burger was solid.",
         '{"mentions": [{"quote": "Super loud inside and the tables were sticky", "polarity": "negative", "intensity": 2}]}'),
        ("Slowest service ever. The waiter forgot our drinks, but the new patio is gorgeous.",
         '{"mentions": [{"quote": "the new patio is gorgeous", "polarity": "positive", "intensity": 3}]}'),
        ("Our waitress was quick and greeted us with a smile. The curry was perfectly spiced.", '{"mentions": []}'),
    ],
}

ASPECT_USER = """<aspect>{aspect}</aspect>
<review>
{review}
</review>
List the {aspect} mentions as JSON."""

AGGREGATOR_SYSTEM = """You are the lead analyst of a team of agents. Three specialists have extracted what a
Yelp reviewer said about food, service and ambience. Read their findings AND the full review
(which may also discuss price, location, parking, etc.), then judge the reviewer's OVERALL
feeling about the business.

  polarity: "positive" | "negative" | "neutral" (neutral = genuinely mixed or lukewarm, "it was okay")
  intensity (ignored when neutral):
    1 = leaning that way, with real reservations
    2 = clearly positive / clearly negative
    3 = enthusiastic / furious
  other_factors: short phrases for anything outside food/service/ambience that influenced the verdict
  rationale: one or two sentences

Return a single JSON object and nothing else:
{"polarity": "...", "intensity": 2, "other_factors": ["..."], "rationale": "..."}"""

AGGREGATOR_USER = """<review>
{review}
</review>
<aspects_json>
{aspects_json}
</aspects_json>
Give the overall verdict as JSON."""

ARBITER_SYSTEM = """You are the arbiter in a team of review-analysis agents. Several specialist agents
cited the SAME passage of a Yelp review as evidence for their aspect. Decide which aspect(s)
the passage genuinely expresses an opinion about. Definitions:
  food: {food}
  service: {service}
  ambience: {ambience}
A passage can belong to more than one aspect only if it explicitly judges each of them.

Return a single JSON object and nothing else:
{{"aspects": ["<aspect>", ...]}}"""

ARBITER_USER = """<review>
{review}
</review>
<quote>{quote}</quote>
<candidates>{candidates}</candidates>
Which of the candidate aspects does this passage express an opinion about?"""

## 4. Quote checking and scoring (pure Python, no LLM)

The agents never output a number. Python checks every quote and computes the scores:

1. **Grounding.** A quote counts only if it is really in the review. Case, quote marks and spaces may differ,
   `...` splits a quote into fragments that are each checked, a near-exact match must cover 85% of a fragment,
   and a quote needs at least 2 words. This cell is the project's `grounding.py`, unchanged.
2. **Mention value** v = sign(polarity) × intensity / 3, so −1, −⅔, −⅓, 0, +⅓, +⅔ or +1.
3. **Aspect score** = the mean value of the aspect's quotes, from −1 to +1. With no quotes the aspect is *not
   mentioned* (score 0); otherwise the label is positive above 0, negative below 0, neutral at exactly 0.
4. **Overall score** = sign × intensity / 3 from the lead agent's verdict, and **rule-based stars** = 3 + 2 × overall.

In [ ]:
_TRANS = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"', "–": "-", "—": "-"})

_ELLIPSIS = re.compile(r"\s*(?:\.\.\.+|…)\s*")

MIN_FRAGMENT_CHARS = 4

MIN_QUOTE_WORDS = 2

FUZZY_COVERAGE = 0.85

def _normalise(text: str) -> tuple[str, list[int]]:
    """Lower-case, unify quotes/dashes, collapse whitespace; keep a map to original offsets."""
    out: list[str] = []
    index: list[int] = []
    prev_space = False
    for i, ch in enumerate(text.translate(_TRANS).lower()):
        if ch.isspace():
            if prev_space:
                continue
            ch, prev_space = " ", True
        else:
            prev_space = False
        out.append(ch)
        index.append(i)
    return "".join(out), index

def _fragments(quote: str) -> list[str]:
    parts = _ELLIPSIS.split(quote.strip())
    cleaned = [p.strip().strip("\"'").strip() for p in parts]
    return [p for p in cleaned if len(p) >= MIN_FRAGMENT_CHARS]

def _find(norm_text: str, frag: str) -> tuple[int, int] | None:
    i = norm_text.find(frag)
    if i >= 0:
        return i, i + len(frag)
    # tolerate a trailing period / comma the model added or dropped
    trimmed = frag.rstrip(".,!?;:")
    if len(trimmed) >= MIN_FRAGMENT_CHARS and (i := norm_text.find(trimmed)) >= 0:
        return i, i + len(trimmed)
    m = SequenceMatcher(None, norm_text, frag, autojunk=False).find_longest_match(0, len(norm_text), 0, len(frag))
    if m.size >= FUZZY_COVERAGE * len(frag):
        start = max(0, m.a - m.b)
        return start, min(len(norm_text), start + len(frag))
    return None

def ground_quote(text: str, quote: str) -> list[list[int]] | None:
    """Return [start, end) spans in *text* for every fragment of *quote*, or None if any is missing."""
    frags = _fragments(quote)
    if not frags or sum(len(f.split()) for f in frags) < MIN_QUOTE_WORDS:
        return None
    norm_text, index = _normalise(text)
    spans: list[list[int]] = []
    for frag in frags:
        norm_frag, _ = _normalise(frag)
        hit = _find(norm_text, norm_frag)
        if hit is None:
            return None
        a, b = hit
        spans.append([index[a], index[b - 1] + 1])
    return spans

In [ ]:
def mention_value(m):
    return POLARITY_SIGN[m["polarity"]] * m["intensity"] / 3


def aspect_score(mentions):
    return round(sum(map(mention_value, mentions)) / len(mentions), 4) if mentions else 0.0


def aspect_label(mentions, score):
    if not mentions:
        return "not_mentioned"
    return "positive" if score > 0 else "negative" if score < 0 else "neutral"


def _overlaps(a, b):
    return any(x0 < y1 and y0 < x1 for x0, x1 in a for y0, y1 in b)


def build_aspect_result(aspect, text, raw_mentions):
    """Validate, ground and de-duplicate one agent's raw mentions, then score them."""
    kept, dropped = [], []
    for raw in raw_mentions or []:
        if not isinstance(raw, dict):
            continue
        polarity = str(raw.get("polarity") or "").strip().lower()
        quote = str(raw.get("quote", "")).strip()
        if polarity not in POLARITY_SIGN or not quote:
            continue
        try:
            intensity = max(1, min(3, int(round(float(raw.get("intensity"))))))
        except (TypeError, ValueError):
            intensity = 1
        m = {"quote": quote[:400], "polarity": polarity, "intensity": intensity, "spans": [], "note": ""}
        spans = ground_quote(text, quote)
        if spans is None:
            m["note"] = "quote not found in review"
            dropped.append(m)
            continue
        m["spans"] = spans
        if any(_overlaps(spans, k["spans"]) for k in kept):  # the same words count once
            continue
        kept.append(m)
    score = aspect_score(kept)
    return {"aspect": aspect, "mentions": kept, "dropped": dropped, "score": score,
            "sentiment": aspect_label(kept, score), "error": None}


def rescore(result):
    result["score"] = aspect_score(result["mentions"])
    result["sentiment"] = aspect_label(result["mentions"], result["score"])


def stars_rule(score):
    return round(3 + 2 * max(-1.0, min(1.0, score)), 3)


def stars_to_label(stars):
    """Star rating -> class; for continuous predictions the cut points are 2.5 and 3.5."""
    return "negative" if stars < 2.5 else "positive" if stars > 3.5 else "neutral"


def build_overall(raw):
    polarity = str(raw.get("polarity") or "").strip().lower()
    if polarity not in POLARITY_SIGN:
        return None
    try:
        intensity = 1 if polarity == "neutral" else max(1, min(3, int(round(float(raw.get("intensity"))))))
    except (TypeError, ValueError):
        intensity = 1
    score = round(POLARITY_SIGN[polarity] * intensity / 3, 4)
    factors = raw.get("other_factors") or []
    return {"polarity": polarity, "intensity": intensity, "score": score, "sentiment": polarity,
            "stars_rule": stars_rule(score),
            "other_factors": [str(f)[:80] for f in factors if f][:6] if isinstance(factors, list) else [],
            "rationale": str(raw.get("rationale", ""))[:500], "error": None}

In [ ]:
# Quick self-checks, taken from the project's tests. They run instantly and need no API.
TEXT = "The pasta was \u201camazing\u201d.   Our waiter was rude and slow. It's loud inside, but cozy."
span_text = lambda spans: [TEXT[a:b] for a, b in spans]
assert span_text(ground_quote(TEXT, 'the pasta was "AMAZING". our')) == ["The pasta was \u201camazing\u201d.   Our"]
assert span_text(ground_quote(TEXT, "The pasta was ... rude and slow")) == ["The pasta was", "rude and slow"]
assert ground_quote(TEXT, "Our waiter was rude and slow!") is not None     # punctuation drift is fine
assert ground_quote(TEXT, "The staff were impolite and sluggish") is None  # a paraphrase is not evidence
assert ground_quote(TEXT, "loud") is None                                  # neither is a single word
m = lambda p, i: {"polarity": p, "intensity": i}
assert aspect_score([m("positive", 2), m("negative", 1)]) == round((2 - 1) / 6, 4)
assert aspect_score([m("negative", 3), m("negative", 3)]) == -1.0
assert stars_rule(1.0) == 5.0 and stars_rule(-1 / 3) == 2.333
print("All checks passed.")

## 5. The agents

Each agent is one chat call in **JSON mode** at temperature 0.1. For every review:

1. The **food, service and ambience agents** run in parallel. Each returns `{"mentions": [{"quote", "polarity", "intensity"}]}`.
2. Python grounds and scores their quotes (section 4).
3. If quotes from two aspects overlap, the **arbiter** decides which aspect the passage belongs to. The losing
   aspect drops those quotes and is re-scored.
4. The **lead agent** reads the review and the kept quotes and returns `{"polarity", "intensity", "other_factors", "rationale"}`.

A failed call never becomes a made-up score. A failed aspect is recorded as an error and counts as not mentioned;
a failed arbiter keeps every claim; a failed lead agent falls back to the mean of the mentioned aspect scores.

In [ ]:
def extract_json(text):
    """The first JSON object in a reply (tolerates ```json fences and extra chatter)."""
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    while start != -1:
        depth = 0
        for i in range(start, len(text)):
            depth += {"{": 1, "}": -1}.get(text[i], 0)
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    break
        start = text.find("{", start + 1)
    raise ValueError(f"no JSON object in model output: {text[:200]!r}")


def chat_json(messages, max_tokens=400):
    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=TEMPERATURE,
                                          max_tokens=max_tokens, response_format={"type": "json_object"})
    return extract_json(resp.choices[0].message.content or "")


def messages(system, shots, user):
    """System prompt, then worked examples as earlier user/assistant turns, then the real request."""
    msgs = [{"role": "system", "content": system}]
    for q, a in shots:
        msgs += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    return msgs + [{"role": "user", "content": user}]


def aspect_agent(aspect, text, few_shot=True):
    system = ASPECT_SYSTEM.format(aspect=aspect, aspect_upper=aspect.upper(),
                                  definition=ASPECT_DEFINITIONS[aspect], rubric=INTENSITY_RUBRIC)
    shots = [(ASPECT_USER.format(aspect=aspect, review=q), a) for q, a in FEW_SHOT[aspect]] if few_shot else []
    try:
        raw = chat_json(messages(system, shots, ASPECT_USER.format(aspect=aspect, review=text)), max_tokens=500)
    except Exception as exc:
        return {"aspect": aspect, "mentions": [], "dropped": [], "score": 0.0, "sentiment": "not_mentioned",
                "error": str(exc)[:300]}
    mentions = raw.get("mentions")
    if mentions is None and "quote" in raw:  # a single mention object instead of a list
        mentions = [raw]
    if not isinstance(mentions, list):
        return {"aspect": aspect, "mentions": [], "dropped": [], "score": 0.0, "sentiment": "not_mentioned",
                "error": f"malformed output: {str(raw)[:200]}"}
    return build_aspect_result(aspect, text, mentions)


def arbiter_agent(text, passage, candidates):
    msgs = messages(ARBITER_SYSTEM.format(**ASPECT_DEFINITIONS), [],
                    ARBITER_USER.format(review=text, quote=passage, candidates=", ".join(candidates)))
    try:
        raw = chat_json(msgs, max_tokens=80)
    except Exception:
        return list(candidates)
    chosen = [a for a in raw.get("aspects", []) if a in candidates] if isinstance(raw.get("aspects"), list) else []
    return chosen or list(candidates)


def lead_agent(text, aspects):
    payload = {a: {"label": r["sentiment"],
                   "mentions": [{k: m[k] for k in ("quote", "polarity", "intensity")} for m in r["mentions"]]}
               for a, r in aspects.items()}
    msgs = messages(AGGREGATOR_SYSTEM, [], AGGREGATOR_USER.format(review=text, aspects_json=json.dumps(payload, indent=1)))
    try:
        overall = build_overall(chat_json(msgs, max_tokens=300))
        if overall:
            return overall
        error = "malformed output"
    except Exception as exc:
        error = str(exc)[:300]
    scores = [r["score"] for r in aspects.values() if r["mentions"]]  # fallback: mean of the mentioned aspects
    score = round(sum(scores) / len(scores), 4) if scores else 0.0
    label = "positive" if score > 0 else "negative" if score < 0 else "neutral"
    return {"polarity": label, "intensity": 0, "score": score, "sentiment": label, "stars_rule": stars_rule(score),
            "other_factors": [], "rationale": "fallback: mean of mentioned aspect scores", "error": error}


def find_conflicts(aspects):
    """Groups of (aspect, mention index) whose quotes overlap across different aspects."""
    nodes = [(a, i) for a, r in aspects.items() for i in range(len(r["mentions"]))]
    group = list(range(len(nodes)))
    def root(x):
        while group[x] != x:
            x = group[x]
        return x
    for i, (a1, m1) in enumerate(nodes):
        for j in range(i + 1, len(nodes)):
            a2, m2 = nodes[j]
            if a1 != a2 and _overlaps(aspects[a1]["mentions"][m1]["spans"], aspects[a2]["mentions"][m2]["spans"]):
                group[root(i)] = root(j)
    groups = defaultdict(list)
    for i, n in enumerate(nodes):
        groups[root(i)].append(n)
    return [g for g in groups.values() if len({a for a, _ in g}) > 1]


def analyze_review(review, few_shot=True, arbitrate=True):
    t0, text = time.perf_counter(), review["text"]
    with cf.ThreadPoolExecutor(max_workers=3) as pool:  # the three aspect agents in parallel
        aspects = dict(zip(ASPECTS, pool.map(lambda a: aspect_agent(a, text, few_shot), ASPECTS)))
    arbitrations = []
    for g in (find_conflicts(aspects) if arbitrate else []):
        spans = [s for a, i in g for s in aspects[a]["mentions"][i]["spans"]]
        passage = text[min(s[0] for s in spans):max(s[1] for s in spans)]
        candidates = sorted({a for a, _ in g}, key=ASPECTS.index)
        keep = arbiter_agent(text, passage, candidates)
        arbitrations.append({"passage": passage, "candidates": candidates, "kept": keep})
        for a in candidates:
            if a in keep:
                continue
            for i in sorted({i for x, i in g if x == a}, reverse=True):
                m = aspects[a]["mentions"].pop(i)
                m["note"] = f"arbiter: passage is about {', '.join(keep)}"
                aspects[a]["dropped"].append(m)
            rescore(aspects[a])
    overall = lead_agent(text, aspects)
    return {"review": review, "aspects": aspects, "overall": overall, "arbitrations": arbitrations,
            "vector": [aspects[a]["score"] for a in ASPECTS], "model": MODEL,
            "few_shot": few_shot, "latency_s": round(time.perf_counter() - t0, 2)}

## 6. One review, step by step

Run the team on one review and look at everything it produced. Leave `CUSTOM_REVIEW` empty to take a review from
the sample, or paste your own text (it then has no real star rating).

In [ ]:
CUSTOM_REVIEW = ""  #@param {type:"string"}
EXAMPLE_INDEX = 0  #@param {type:"integer"}

review = ({"review_id": "custom", "business_id": "custom", "stars": None, "text": CUSTOM_REVIEW,
           "business_name": "(your review)", "city": "", "date": ""}
          if CUSTOM_REVIEW.strip() else accuracy_sample[EXAMPLE_INDEX])
one = analyze_review(review, few_shot=USE_FEW_SHOT)


def show_analysis(res):
    r, o = res["review"], res["overall"]
    stars = "" if r["stars"] is None else f" \u00b7 reviewer gave {r['stars']:.0f}\u2605"
    display(Markdown(f"**{r['business_name']}** {r.get('city', '')}{stars}\n\n> {r['text']}"))
    rows = [{"aspect": a, "quote": m["quote"], "polarity": m["polarity"], "intensity": m["intensity"],
             "value": round(mention_value(m), 2), "kept": "yes" if not m.get("note") else m["note"]}
            for a in ASPECTS for m in res["aspects"][a]["mentions"] + res["aspects"][a]["dropped"]]
    if rows:
        display(pd.DataFrame(rows))
    for a in ASPECTS:
        x = res["aspects"][a]
        if x["error"]:
            print(f"{a:9} ERROR {x['error']}")
        elif not x["mentions"]:
            print(f"{a:9} not mentioned (score 0)")
        else:
            terms = " ".join(f"{POLARITY_SIGN[m['polarity']] * m['intensity']:+d}" for m in x["mentions"])
            print(f"{a:9} ({terms}) / (3 \u00d7 {len(x['mentions'])}) = {x['score']:+.2f}  -> {x['sentiment']}")
    for arb in res["arbitrations"]:
        print(f"arbiter: \"{arb['passage'][:80]}\" claimed by {', '.join(arb['candidates'])} -> kept for {', '.join(arb['kept'])}")
    print(f"lead agent: {o['polarity']}, intensity {o['intensity']}, other factors {o['other_factors']}")
    print(f"  overall {o['score']:+.2f} -> stars 3 + 2 \u00d7 {o['score']:+.2f} = {o['stars_rule']:.2f}")
    print(f"  rationale: {o['rationale']}")
    print(f"vector [food, service, ambience] = {[round(v, 2) for v in res['vector']]} \u00b7 {res['latency_s']} s")


show_analysis(one)

## 7. Accuracy check: do the agents agree with the stars?

`run_batch` analyses reviews in parallel and appends each result to a JSONL file as soon as it finishes. If the
session disconnects, run the cell again: reviews already in the file are skipped. Each model writes its own file,
so switching `MODEL` never mixes results.

The only ground truth is the star rating: 1–2★ counts as negative, 3★ as neutral, 4–5★ as positive.

In [ ]:
def load_results(path):
    """Latest result per review; a half-written last line (an interrupted run) is skipped."""
    done = {}
    if Path(path).exists():
        for line in Path(path).read_text().splitlines():
            try:
                res = json.loads(line)
            except json.JSONDecodeError:
                continue
            done[res["review"]["review_id"]] = res
    return done


def results_file(kind):
    """One file per sample, model and prompt variant, so different runs never mix."""
    return OUT_DIR / f"{kind}_results_{MODEL}{'' if USE_FEW_SHOT else '_zero-shot'}.jsonl"


def run_batch(reviews, out_path, workers=WORKERS):
    done = load_results(out_path)
    todo = [r for r in reviews if r["review_id"] not in done]
    print(f"{len(reviews) - len(todo)} already done, {len(todo)} to analyse with {MODEL} -> {out_path.name}")
    with open(out_path, "a") as f, cf.ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(analyze_review, r, USE_FEW_SHOT) for r in todo]
        for fut in tqdm(cf.as_completed(futures), total=len(futures), desc="reviews"):
            res = fut.result()
            f.write(json.dumps(res) + "\n")
            f.flush()
            done[res["review"]["review_id"]] = res
    return [done[r["review_id"]] for r in reviews if r["review_id"] in done]


limit = MAX_REVIEWS_TO_ANALYZE or None
results = run_batch(accuracy_sample[:limit], results_file("accuracy"))

In [ ]:
def true_label(stars):
    return "negative" if stars <= 2 else "positive" if stars >= 4 else "neutral"


def wilson(k, n, z=1.959964):
    """95% Wilson interval for a proportion k/n; stays inside 0-1 even for small n."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (max(0.0, centre - half), min(1.0, centre + half))


def bootstrap_macro_f1(y_true, y_pred, rounds=2000, seed=0):
    rng, t, p = np.random.default_rng(seed), np.array(y_true), np.array(y_pred)
    scores = [f1_score(t[i], p[i], labels=CLASSES, average="macro", zero_division=0)
              for i in (rng.integers(0, len(t), len(t)) for _ in range(rounds))]
    return tuple(np.percentile(scores, [2.5, 97.5]))


def evaluate(results):
    y_true = [true_label(r["review"]["stars"]) for r in results]
    y_pred = [r["overall"]["sentiment"] for r in results]  # note the order: (y_true, y_pred)
    stars = np.array([r["review"]["stars"] for r in results])
    pred = np.array([r["overall"]["stars_rule"] for r in results])
    k = sum(a == b for a, b in zip(y_true, y_pred))
    lo, hi = wilson(k, len(results))
    f_lo, f_hi = bootstrap_macro_f1(y_true, y_pred)
    errors = sum(any(a["error"] for a in r["aspects"].values()) or bool(r["overall"]["error"]) for r in results)
    print(f"{len(results)} reviews, {errors} with agent errors, mean {np.mean([r['latency_s'] for r in results]):.1f} s per review")
    print(f"Accuracy {k / len(results):.1%} (95% CI {lo:.0%}-{hi:.0%})   "
          f"macro-F1 {f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0):.3f} "
          f"(95% CI {f_lo:.2f}-{f_hi:.2f})")
    print(f"Rule-based stars: MAE {mean_absolute_error(stars, pred):.3f}, RMSE {math.sqrt(mean_squared_error(stars, pred)):.3f}, "
          f"Pearson r {stats.pearsonr(pred, stars).statistic:.3f}, Spearman \u03c1 {stats.spearmanr(pred, stars).statistic:.3f}\n")
    print(classification_report(y_true, y_pred, labels=CLASSES, zero_division=0))
    cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=CLASSES),
                      index=[f"actual {c}" for c in CLASSES], columns=[f"predicted {c}" for c in CLASSES])
    display(cm.style.background_gradient(cmap="Blues"))
    rows = []
    for a in ASPECTS:
        said = [r for r in results if r["aspects"][a]["mentions"]]
        kept = sum(len(r["aspects"][a]["mentions"]) for r in results)
        missing = sum(m["note"] == "quote not found in review" for r in results for m in r["aspects"][a]["dropped"])
        rho = stats.spearmanr([r["aspects"][a]["score"] for r in said], [r["review"]["stars"] for r in said]).statistic \
            if len(said) >= 3 else float("nan")
        rows.append({"aspect": a, "mentioned": f"{len(said) / len(results):.0%}",
                     "mean score": round(np.mean([r["aspects"][a]["score"] for r in said]), 2) if said else None,
                     "Spearman \u03c1 vs stars": round(rho, 2), "quotes kept": kept, "quotes not found": missing})
    display(pd.DataFrame(rows).set_index("aspect"))


evaluate(results)

**Calibrated stars.** The rule-based stars use only the lead agent's verdict. A ridge regression can instead
*learn* how much each aspect matters, from 7 features: the three aspect scores, three "was it mentioned" flags
(so *not mentioned* and *neutral* don't look the same), and the overall score. It is scored with 5-fold
cross-validation, so every prediction comes from a model that never saw that review.

In [ ]:
def features(r):
    return ([r["aspects"][a]["score"] for a in ASPECTS]
            + [1.0 if r["aspects"][a]["mentions"] else 0.0 for a in ASPECTS] + [r["overall"]["score"]])


FEATURES = [*ASPECTS, *(f"{a}_mentioned" for a in ASPECTS), "overall"]
X = np.array([features(r) for r in results])
y = np.array([r["review"]["stars"] for r in results])
if len(results) >= 10 and len(set(y)) >= 2:
    folds = KFold(min(5, len(results)), shuffle=True, random_state=0)
    oof = np.clip(cross_val_predict(Ridge(alpha=1.0), X, y, cv=folds), 1, 5)
    rule = np.array([r["overall"]["stars_rule"] for r in results])
    print(f"MAE  rule {mean_absolute_error(y, rule):.3f}   calibrated {mean_absolute_error(y, oof):.3f}")
    print(f"RMSE rule {math.sqrt(mean_squared_error(y, rule)):.3f}   calibrated {math.sqrt(mean_squared_error(y, oof)):.3f}")
    model = Ridge(alpha=1.0).fit(X, y)
    display(pd.Series(dict(zip(FEATURES, model.coef_)), name="stars per unit").round(2).to_frame())
else:
    print("Needs at least 10 results with 2 different star ratings.")

In [ ]:
# Each review as a point in 3-D aspect space, coloured by the reviewer's own rating. Drag to rotate.
import plotly.express as px

COLORS = {"negative": "#e34948", "neutral": "#8a8984", "positive": "#2a78d6"}  # the project's diverging palette
SYMBOLS = {"negative": "square", "neutral": "diamond", "positive": "circle"}     # shape too, never colour alone
jitter = lambda s, k: zlib.crc32((s + k).encode()) % 1000 / 1000 * 0.06 - 0.03  # many reviews share 0 on an axis
frame = pd.DataFrame([{"food": r["vector"][0] + jitter(r["review"]["review_id"], "f"),
                       "service": r["vector"][1] + jitter(r["review"]["review_id"], "s"),
                       "ambience": r["vector"][2] + jitter(r["review"]["review_id"], "a"),
                       "actual": true_label(r["review"]["stars"]), "stars": r["review"]["stars"],
                       "business": r["review"]["business_name"]} for r in results])
fig = px.scatter_3d(frame, x="food", y="service", z="ambience", color="actual", symbol="actual",
                    color_discrete_map=COLORS, symbol_map=SYMBOLS, hover_data=["business", "stars"],
                    category_orders={"actual": CLASSES}, range_x=[-1.1, 1.1], range_y=[-1.1, 1.1], range_z=[-1.1, 1.1],
                    title="Reviews in aspect space, coloured by the reviewer's star class")
fig.update_traces(marker_size=4)
fig.show()

## 8. Restaurant comparison: do reviewers of the same restaurant agree?

Now the restaurant sample: every restaurant has 10 reviews, analysed exactly as above. For each restaurant and
aspect, only the reviews that mention the aspect count:

| Measure | Definition |
| --- | --- |
| Agreement | Share of those reviews on the most common side (positive, neutral or negative); needs at least 2 mentions |
| Verdict | That side if agreement is at least 75%, otherwise **mixed**; **too few** with fewer than 2 mentions |
| Contested | At least one positive and at least one negative review |

Across restaurants, a one-way ANOVA and **ICC(1)** ask whether knowing *which restaurant* a review is about tells
you anything about its score. ICC(1) is the share of score variance that lies between restaurants: 0 means
reviewers of the same place agree no more than strangers, 1 means they agree completely. The reviewers' own
stars get the same treatment as a baseline.

The reviews are stored grouped by restaurant, so `MAX_REVIEWS_TO_ANALYZE = 30` analyses 3 whole restaurants.

In [ ]:
restaurant_results = run_batch(restaurant_sample[:limit], results_file("restaurant"))

In [ ]:
CONSENSUS, MIN_COMPARE = 0.75, 2


def one_way(groups):
    """One-way ANOVA (F, p) and ICC(1) over the groups with at least 2 values."""
    groups = [list(map(float, g)) for g in groups if len(g) >= 2]
    k, n = len(groups), sum(map(len, groups))
    out = {"F": None, "p": None, "icc1": None}
    if k < 2 or n <= k:
        return out
    grand = np.mean([x for g in groups for x in g])
    msb = sum(len(g) * (np.mean(g) - grand) ** 2 for g in groups) / (k - 1)
    msw = sum(((np.asarray(g) - np.mean(g)) ** 2).sum() for g in groups) / (n - k)
    n0 = (n - sum(len(g) ** 2 for g in groups) / n) / (k - 1)  # mean group size, adjusted for unequal groups
    if msw > 0:
        out.update(F=msb / msw, p=float(stats.f.sf(msb / msw, k - 1, n - k)))
    if msb + (n0 - 1) * msw > 0:
        out["icc1"] = (msb - msw) / (msb + (n0 - 1) * msw)
    return out


def aspect_block(rs, a):
    said = [r for r in rs if r["aspects"][a]["mentions"] and not r["aspects"][a]["error"]]
    scores = [r["aspects"][a]["score"] for r in said]
    labels = Counter(r["aspects"][a]["sentiment"] for r in said)
    side, count = max(((s, labels.get(s, 0)) for s in ("positive", "neutral", "negative")), key=lambda x: x[1])
    agreement = count / len(said) if len(said) >= MIN_COMPARE else None
    verdict = "too few" if agreement is None else side if agreement >= CONSENSUS else "mixed"
    quotes = [(mention_value(m), m["quote"], r["review"]["stars"]) for r in said for m in r["aspects"][a]["mentions"]]
    best, worst = max(quotes, default=None), min(quotes, default=None)
    return {"mentioned": len(said), "mean": np.mean(scores) if scores else None,
            "sd": np.std(scores, ddof=1) if len(scores) >= 2 else None, "agreement": agreement, "verdict": verdict,
            "contested": labels["positive"] > 0 and labels["negative"] > 0, "scores": scores,
            "most_positive": best if best and best[0] > 0 else None,
            "most_negative": worst if worst and worst[0] < 0 else None}


groups = defaultdict(list)
for r in restaurant_results:
    groups[r["review"]["business_id"]].append(r)
groups = {b: rs for b, rs in groups.items() if len(rs) >= 2}

restaurants = []
for bid, rs in groups.items():
    rv = rs[0]["review"]
    restaurants.append({"business_id": bid, "restaurant": rv["business_name"], "city": rv["city"],
                        "yelp_stars": rv.get("business_stars"), "reviews": len(rs),
                        "sampled_stars": np.mean([r["review"]["stars"] for r in rs]),
                        "predicted_stars": np.mean([r["overall"]["stars_rule"] for r in rs]),
                        **{a: aspect_block(rs, a) for a in ASPECTS}})

summary = []
for a in ASPECTS:
    blocks = [x[a] for x in restaurants]
    compared = [b for b in blocks if b["mentioned"] >= MIN_COMPARE]
    means = [(b["mean"], x["yelp_stars"]) for x, b in zip(restaurants, blocks) if b["mean"] is not None and x["yelp_stars"]]
    aov = one_way([b["scores"] for b in blocks])
    summary.append({"aspect": a, "restaurants compared": len(compared),
                    "within-restaurant sd": np.mean([b["sd"] for b in compared]) if compared else None,
                    "between-restaurant sd": np.std([b["mean"] for b in blocks if b["mean"] is not None], ddof=1)
                    if sum(b["mean"] is not None for b in blocks) >= 2 else None,
                    "ICC(1)": aov["icc1"], "ANOVA p": aov["p"], "contested": sum(b["contested"] for b in blocks),
                    "verdicts": dict(Counter(b["verdict"] for b in compared)),
                    "\u03c1 vs Yelp rating": stats.spearmanr(*zip(*means)).statistic if len(means) >= 3 else None})
base = one_way([[r["review"]["stars"] for r in rs] for rs in groups.values()])
summary.append({"aspect": "reviewer stars (baseline)", "restaurants compared": len(groups),
                "ICC(1)": base["icc1"], "ANOVA p": base["p"]})
display(pd.DataFrame(summary).set_index("aspect").style.format(precision=3, na_rep="")
        .format({"restaurants compared": "{:.0f}", "contested": "{:.0f}"}, na_rep=""))

if restaurants:
    single = np.mean([abs(r["overall"]["stars_rule"] - r["review"]["stars"]) for rs in groups.values() for r in rs])
    averaged = np.mean([abs(x["predicted_stars"] - x["sampled_stars"]) for x in restaurants])
    print(f"Predicted stars miss a single review by {single:.2f} stars on average, "
          f"and a restaurant's average of its reviews by {averaged:.2f}.")

In [ ]:
# One row per restaurant: mean score, agreement and verdict per aspect. Most contested first.
cell = lambda b: "\u2014" if b["mean"] is None else (
    f"{b['mean']:+.2f} {b['verdict']}" + ("" if b["agreement"] is None else f" ({b['agreement']:.0%})"))
table = pd.DataFrame([{"restaurant": x["restaurant"], "city": x["city"], "Yelp \u2605": x["yelp_stars"],
                       "sampled \u2605": round(x["sampled_stars"], 1), "predicted \u2605": round(x["predicted_stars"], 1),
                       **{a: cell(x[a]) for a in ASPECTS},
                       "reviewers disagree on": ", ".join(a for a in ASPECTS if x[a]["contested"]) or "\u2014"}
                      for x in restaurants])
table["n_contested"] = [sum(x[a]["contested"] for a in ASPECTS) for x in restaurants]
table.sort_values(["n_contested", "restaurant"], ascending=[False, True]).drop(columns="n_contested").reset_index(drop=True)

In [ ]:
def show_restaurant(name):
    """One restaurant's reviews side by side, with the quotes that pull furthest apart."""
    x = next((x for x in restaurants if name.lower() in x["restaurant"].lower()), None)
    if x is None:
        print("No restaurant matching", repr(name))
        return
    yelp = "n/a" if x["yelp_stars"] is None else f"{x['yelp_stars']}\u2605"
    display(Markdown(f"### {x['restaurant']} ({x['city']})\nYelp rating {yelp} \u00b7 these {x['reviews']} reviews average "
                     f"{x['sampled_stars']:.1f}\u2605 \u00b7 the lead agent predicts {x['predicted_stars']:.1f}\u2605"))
    for a in ASPECTS:
        b = x[a]
        mean = "" if b["mean"] is None else f", mean {b['mean']:+.2f}"
        print(f"{a:9} {b['verdict']:8} {b['mentioned']} of {x['reviews']} mention it{mean}")
        for q in (b["most_positive"], b["most_negative"]):
            if q:
                print(f"            {q[0]:+.2f}  \u201c{q[1][:90]}\u201d  ({q[2]:.0f}\u2605 review)")
    rs = sorted(groups[x["business_id"]], key=lambda r: r["review"]["date"])
    display(pd.DataFrame([{"date": r["review"]["date"][:10], "\u2605": r["review"]["stars"],
                           "predicted \u2605": r["overall"]["stars_rule"],
                           **{a: (round(r["aspects"][a]["score"], 2) if r["aspects"][a]["mentions"] else None) for a in ASPECTS},
                           "review": r["review"]["text"][:120]} for r in rs]))


RESTAURANT = ""  #@param {type:"string"}
show_restaurant(RESTAURANT or (restaurants[0]["restaurant"] if restaurants else ""))

In [ ]:
# A restaurant's mean aspect score against its overall Yelp rating, one panel per aspect.
points = [(x["yelp_stars"], x[a]["mean"], a) for x in restaurants for a in ASPECTS
          if x["yelp_stars"] is not None and x[a]["mean"] is not None]
if len({p[0] for p in points}) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)
    for ax, a in zip(axes, ASPECTS):
        pts = [(s, m) for s, m, k in points if k == a]
        xs = [s + random.Random(i).uniform(-0.12, 0.12) for i, (s, _) in enumerate(pts)]  # ratings are in half stars
        ax.scatter(xs, [m for _, m in pts], s=28, alpha=0.85, color="#2a78d6" if a == "food" else "#a3a29c")
        rho = stats.spearmanr([m for _, m in pts], [s for s, _ in pts]).statistic if len(pts) >= 3 else float("nan")
        rho_text = f"\u03c1 = {rho:.2f}" if not math.isnan(rho) else "\u03c1 needs 3+ restaurants"
        ax.set_title(f"{a.capitalize()} \u00b7 {rho_text}", loc="left", fontsize=11, fontweight="bold")
        ax.axhline(0, color="#c3c2b7", linewidth=1)
        ax.grid(color="#e1e0d9", linewidth=0.6)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.set_xlabel("Restaurant's Yelp rating (stars)")
    axes[0].set_ylabel("Mean aspect score of its sampled reviews")
    axes[0].set_ylim(-1.05, 1.05)
    axes[0].set_xlim(0.7, 5.3)
    axes[0].set_xticks([1, 2, 3, 4, 5])
    fig.suptitle("Do a restaurant's aspect scores follow its Yelp rating?", x=0.01, ha="left", fontsize=13)
    fig.tight_layout()
    plt.show()
else:
    print("Needs restaurants with at least two different Yelp ratings (the demo reviews have none).")

## 9. Exercises

Each needs only a change to a setting above, and every run writes to its own results file.

1. **Do the few-shot examples help?** Untick `USE_FEW_SHOT` in section 1 and re-run section 7. The results go to a
   separate `_zero-shot` file; compare macro-F1 and the recall for 3★ reviews with the few-shot run.
2. **Does a bigger model read 3★ reviews better?** Set `MODEL = "qwen3-235b-a22b-instruct-2507"`, re-run sections 7
   and 8, and compare the confusion matrices.
3. **How accurate are the aspects themselves?** Hand-label the food, service and ambience polarity of 30 reviews and
   compare your labels with the agents'. Where do *you* disagree with each other?
4. **Are the results stable?** Run the same reviews twice into two files. How many aspect labels change?
5. **More reviews per restaurant.** Set `REVIEWS_PER_RESTAURANT = 20`. Do ICC(1) and the number of *mixed* verdicts move?
6. **Is price the missing aspect?** It is the lead agent's most common outside factor. Add a `"price"` entry to
   `ASPECT_DEFINITIONS`, `FEW_SHOT` and `ASPECTS`, update the arbiter and lead prompts, and see whether it improves
   the calibrated stars.